# RecoMart — Data Preparation & EDA (Task 5)

This notebook demonstrates the **data cleaning / preprocessing** and **exploratory data analysis** performed on the RecoMart interaction data. It reuses the production pipeline code in `src/` so the notebook and the automated pipeline stay consistent.

**Contents**
1. Setup
2. Pipeline layers (Bronze -> Silver -> Gold) and cleaning logic
3. Data quality validation report
4. Descriptive profiling (size, interaction totals, sparsity)
5. Interaction distributions, popularity, and co-occurrence plots

## 1. Setup

Add the repository root to the path and point at the SQLite database produced by the ingestion + curation pipeline.

In [2]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.core import connect, table_counts
from src.validation import validate
from src.modeling import profile_gold, generate_eda_plots

DB_PATH = ROOT / "data" / "recomart.db"
assert DB_PATH.exists(), (
    f"{DB_PATH} not found. Build it first, e.g. `python -m src.recomart run`."
)
DB_PATH

PosixPath('/Users/nitheesh/personal/repos/neavepaul/recomart-recommender/data/recomart.db')

## 2. Pipeline layers and cleaning logic

The medallion layers apply the cleaning and preprocessing steps:

- **Bronze** — raw ingested events, item properties, and category tree.
- **Silver** — cleaned data: invalid event types and negative/zero IDs and timestamps are filtered out, timestamps are normalised to ISO format, and only the *latest* value per item property is retained (deduplication).
- **Gold** — model-ready features: user-item interactions are aggregated into weighted interaction scores (view=1, addtocart=3, transaction=5) and categorical item properties are encoded into sparse hashed feature vectors.

The row counts below show the effect of each layer.

In [ ]:
db = connect(DB_PATH)
try:
    counts = table_counts(db)
finally:
    db.close()

for name, count in counts.items():
    print(f"{name:<32} {count:>12,}")

## 3. Data quality validation report

`validate()` runs automated checks: valid event types only, product/user-item uniqueness (no duplicates), the interaction-score formula, and vector completeness.

In [ ]:
report = validate(DB_PATH)
print(json.dumps(report, indent=2))
assert report["ok"], "Data quality checks failed"

## 4. Descriptive profiling

`profile_gold()` summarises dataset size, interaction totals, sparsity, and the per-user / per-item interaction distributions.

In [ ]:
profile = profile_gold(DB_PATH, top_n=15)

print("Size")
for key, value in profile["size"].items():
    print(f"  {key:<20} {value:>12,}")

print("\nInteraction totals")
for key, value in profile["interaction_totals"].items():
    print(f"  {key:<20} {value:>12,}")

print("\nSparsity")
for key, value in profile["sparsity"].items():
    print(f"  {key:<28} {value:,.6f}" if isinstance(value, float) else f"  {key:<28} {value:,}")

In [ ]:
print("Items per user (interaction count percentiles)")
for key, value in profile["items_per_user"].items():
    print(f"  {key:<6} {value:,.2f}" if isinstance(value, float) else f"  {key:<6} {value:,}")

print("\nUsers per item (interaction count percentiles)")
for key, value in profile["users_per_item"].items():
    print(f"  {key:<6} {value:,.2f}" if isinstance(value, float) else f"  {key:<6} {value:,}")

## 5. Summary plots (histograms & heatmap)

`generate_eda_plots()` renders the summary plots to `reports/eda/` and returns their paths. We display them inline below.

In [ ]:
plots = generate_eda_plots(DB_PATH, out_dir=ROOT / "reports" / "eda", top_n=15)
print(json.dumps(plots, indent=2))

In [ ]:
from IPython.display import Image, display

for name, path in plots["plots"].items():
    print(name)
    display(Image(filename=path))